<a href="https://colab.research.google.com/github/fedecasart/CienciaDatosDelitosTPO/blob/main/delitos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the dataset
df_delitos = pd.read_csv('/content/sample_data/delitos_2023.csv')

First, let's inspect the columns to identify potential geolocation columns and check for missing values.

In [ ]:
# Display basic info about the DataFrame to identify geolocation columns
display(df_delitos.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155897 entries, 0 to 155896
Data columns (total 15 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   id-mapa   155897 non-null  int64  
 1   anio      155897 non-null  int64  
 2   mes       155897 non-null  object 
 3   dia       155897 non-null  object 
 4   fecha     155897 non-null  object 
 5   franja    155897 non-null  int64  
 6   tipo      155897 non-null  object 
 7   subtipo   155897 non-null  object 
 8   uso_arma  155897 non-null  object 
 9   uso_moto  155897 non-null  object 
 10  barrio    152739 non-null  object 
 11  comuna    152739 non-null  float64
 12  latitud   155549 non-null  float64
 13  longitud  155549 non-null  float64
 14  cantidad  155897 non-null  int64  
dtypes: float64(3), int64(4), object(8)
memory usage: 17.8+ MB


None

Based on the column names, 'Latitud' and 'Longitud' appear to be the geolocation columns. Now, I will separate the DataFrame into two based on the presence of values in these columns.

In [ ]:
import numpy as np

# Define a function to normalize coordinates for Buenos Aires
def normalize_ba_coords(coord, coord_type):
    if pd.isna(coord):
        return coord

    # Approximate valid ranges for Buenos Aires
    lat_min_ba, lat_max_ba = -35.0, -34.0
    lon_min_ba, lon_max_ba = -59.0, -58.0

    # Check if already within a reasonable range
    if coord_type == 'lat':
        if lat_min_ba <= coord <= lat_max_ba:
            return coord
    elif coord_type == 'lon':
        if lon_min_ba <= coord <= lon_max_ba:
            return coord

    # Try various common scaling factors if coord is 'large' and outside range
    # Ordered from most common expected to less common/larger scaling
    potential_divisors = [1_000_000.0, 10_000_000.0, 100_000.0, 100_000_000.0]

    for divisor in potential_divisors:
        # Only attempt scaling if the absolute value is significantly larger than 100
        # to avoid incorrectly scaling already valid small numbers
        if abs(coord) > 100.0:
            scaled_coord = coord / divisor
            if coord_type == 'lat':
                if lat_min_ba <= scaled_coord <= lat_max_ba:
                    return scaled_coord
            elif coord_type == 'lon':
                if lon_min_ba <= scaled_coord <= lon_max_ba:
                    return scaled_coord

    # If no scaling factor works or if the number was small and outside the range,
    # or if it was an extremely large value that didn't normalize, return NaN
    return np.nan

# Apply the normalization function to the original DataFrame
df_delitos['latitud_normalized'] = df_delitos['latitud'].apply(lambda x: normalize_ba_coords(x, 'lat'))
df_delitos['longitud_normalized'] = df_delitos['longitud'].apply(lambda x: normalize_ba_coords(x, 'lon'))

# Define the new geolocation columns for filtering
geoloc_columns_normalized = ['latitud_normalized', 'longitud_normalized']

# Filter rows where both normalized geolocation columns are not null
df_with_geolocation = df_delitos.dropna(subset=geoloc_columns_normalized).copy()

# Filter rows where at least one normalized geolocation column is null
df_without_geolocation = df_delitos[df_delitos[geoloc_columns_normalized].isnull().any(axis=1)].copy()

print("\n--- DataFrame with Geolocation (Normalized) ---")
display(df_with_geolocation[['latitud', 'longitud', 'latitud_normalized', 'longitud_normalized']].head())
print(f"Shape of DataFrame with normalized geolocation: {df_with_geolocation.shape}")

print("\n--- DataFrame without Geolocation (including un-normalizable) ---")
display(df_without_geolocation[['latitud', 'longitud', 'latitud_normalized', 'longitud_normalized']].head())
print(f"Shape of DataFrame without normalized geolocation: {df_without_geolocation.shape}")


--- DataFrame with Geolocation (Normalized) ---


,latitud,longitud,latitud_normalized,longitud_normalized
469,-3.472625e+09,-582836498.0,-34.726248,-58.283650
470,-3.462512e+07,-58348646.0,-34.625121,-58.348646
471,-3.462512e+07,-58348646.0,-34.625121,-58.348646
472,-3.462512e+07,-58348646.0,-34.625121,-58.348646
473,-3.463364e+07,-58353495.0,-34.633645,-58.353495


Shape of DataFrame with normalized geolocation: (149852, 17)

--- DataFrame without Geolocation (including un-normalizable) ---


,latitud,longitud,latitud_normalized,longitud_normalized
0,0.0,0.0,NaN,NaN
1,0.0,0.0,NaN,NaN
2,0.0,0.0,NaN,NaN
3,0.0,0.0,NaN,NaN
4,0.0,0.0,NaN,NaN


Shape of DataFrame without normalized geolocation: (6045, 17)


Let's investigate the range and distribution of the `latitud` and `longitud` columns in `df_with_geolocation` to understand the non-standardized format.

### Inspección de las Coordenadas en `comisarias_policia.csv`

Vamos a cargar el dataset de comisarías y examinar sus columnas para identificar los datos de geolocalización.

In [ ]:
# Load the comisarias_policia.csv dataset
df_comisarias = pd.read_csv('/content/sample_data/comisarias_policia.csv')

print("--- Información del DataFrame de Comisarías ---")
display(df_comisarias.info())

print("\n--- Primeras filas del DataFrame de Comisarías ---")
display(df_comisarias.head())

--- Información del DataFrame de Comisarías ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75 entries, 0 to 74
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          75 non-null     int64  
 1   nombre      75 non-null     object 
 2   calle       75 non-null     object 
 3   altura      74 non-null     float64
 4   direccion   75 non-null     object 
 5   telefonos   75 non-null     int64  
 6   barrio      75 non-null     object 
 7   comuna      75 non-null     int64  
 8   codigo_pos  61 non-null     object 
 9   geometry    75 non-null     object 
dtypes: float64(1), int64(3), object(6)
memory usage: 6.0+ KB


None


--- Primeras filas del DataFrame de Comisarías ---


,id,nombre,calle,altura,direccion,telefonos,barrio,comuna,codigo_pos,geometry
0,1,Comisaría Vecinal12-A (edificio anexo),RAMALLO,4398.0,RAMALLO 4398,911,Saavedra,12,NaN,POINT (-58.49101503190442 -34.550990102464745)
1,2,Comisaría Vecinal 7-A,"BONORINO, ESTEBAN, CNEL. AV.",258.0,"BONORINO, ESTEBAN, CNEL. AV. 258",911,Flores,7,C1406DME,POINT (-58.458307591055934 -34.6310563022503)
2,3,Comisaría Vecinal 6-B,AVELLANEDA AV.,1548.0,AVELLANEDA AV. 1548,911,Caballito,6,NaN,POINT (-58.45322168166576 -34.62038139407484)
3,4,Comisaría Vecinal 5-A,BILLINGHURST,471.0,BILLINGHURST 471,911,Almagro,5,C1174ABG,POINT (-58.415664541274616 -34.604406411773645)
4,5,Comisaría Vecinal 4-A,ZAVALETA,425.0,ZAVALETA 425,911,Parque Patricios,4,NaN,POINT (-58.40283281867738 -34.64194792387007)


Parece que las columnas de geolocalización en `df_comisarias` son `lat` y `lng`. Ahora aplicaremos la misma función `normalize_ba_coords` para ver si estos datos requieren normalización.

In [ ]:
# Extract latitude and longitude from the 'geometry' column
# The format is 'POINT (longitude latitude)'
coords = df_comisarias['geometry'].str.extract(r'POINT \((?P<longitude>[-]?\d+\.?\d*)\s(?P<latitude>[-]?\d+\.?\d*)\)')
df_comisarias['extracted_lat'] = pd.to_numeric(coords['latitude'])
df_comisarias['extracted_lon'] = pd.to_numeric(coords['longitude'])

# Apply the normalization function to the extracted coordinates
df_comisarias['lat_normalized'] = df_comisarias['extracted_lat'].apply(lambda x: normalize_ba_coords(x, 'lat'))
df_comisarias['lng_normalized'] = df_comisarias['extracted_lon'].apply(lambda x: normalize_ba_coords(x, 'lon'))

# Filter rows where both normalized geolocation columns are not null
df_comisarias_with_geolocation = df_comisarias.dropna(subset=['lat_normalized', 'lng_normalized']).copy()

# Filter rows where at least one normalized geolocation column is null
df_comisarias_without_geolocation = df_comisarias[df_comisarias[['lat_normalized', 'lng_normalized']].isnull().any(axis=1)].copy()

print("\n--- DataFrame de Comisarías con Geolocalización (Normalizada) ---")
display(df_comisarias_with_geolocation[['extracted_lat', 'extracted_lon', 'lat_normalized', 'lng_normalized']].head())
print(f"Shape of DataFrame con geolocalización normalizada: {df_comisarias_with_geolocation.shape}")

print("\n--- DataFrame de Comisarías sin Geolocalización (incluyendo no normalizables) ---")
display(df_comisarias_without_geolocation[['extracted_lat', 'extracted_lon', 'lat_normalized', 'lng_normalized']].head())
print(f"Shape of DataFrame sin geolocalización normalizada: {df_comisarias_without_geolocation.shape}")


--- DataFrame de Comisarías con Geolocalización (Normalizada) ---


,extracted_lat,extracted_lon,lat_normalized,lng_normalized
0,-34.550990,-58.491015,-34.550990,-58.491015
1,-34.631056,-58.458308,-34.631056,-58.458308
2,-34.620381,-58.453222,-34.620381,-58.453222
3,-34.604406,-58.415665,-34.604406,-58.415665
4,-34.641948,-58.402833,-34.641948,-58.402833


Shape of DataFrame con geolocalización normalizada: (75, 14)

--- DataFrame de Comisarías sin Geolocalización (incluyendo no normalizables) ---


,extracted_lat,extracted_lon,lat_normalized,lng_normalized


Shape of DataFrame sin geolocalización normalizada: (0, 14)


### Calcular Comisaría Más Cercana y Distancia

Ahora que tenemos los datos de delitos y comisarías con coordenadas estandarizadas, podemos calcular la comisaría más cercana para cada delito y la distancia a la misma.

In [ ]:
from geopy.distance import geodesic

# Prepare comisarias locations as a list of tuples (name, (latitude, longitude))
# We use the normalized coordinates from df_comisarias_with_geolocation
comisarias_locations = [
    (row['nombre'], (row['lat_normalized'], row['lng_normalized']))
    for index, row in df_comisarias_with_geolocation.iterrows()
]

def find_closest_comisaria(crime_lat, crime_lon):
    if pd.isna(crime_lat) or pd.isna(crime_lon):
        return np.nan, np.nan

    crime_coords = (crime_lat, crime_lon)
    min_distance = float('inf')
    closest_comisaria_name = None

    for comisaria_name, comisaria_coords in comisarias_locations:
        distance = geodesic(crime_coords, comisaria_coords).meters
        if distance < min_distance:
            min_distance = distance
            closest_comisaria_name = comisaria_name

    return closest_comisaria_name, min_distance

# Apply the function to the df_with_geolocation DataFrame
# This operation can be time-consuming due to iterating through all comisarias for each crime.
print("Calculando la comisaría más cercana y la distancia para cada delito... Esto puede tardar varios minutos.")

results = df_with_geolocation.apply(
    lambda row: find_closest_comisaria(row['latitud_normalized'], row['longitud_normalized']),
    axis=1,
    result_type='expand'
)

df_with_geolocation['comisaria_mas_cercana'] = results[0]
df_with_geolocation['distancia_a_comisaria_metros'] = results[1]

print("Cálculo completado. Mostrando las primeras filas con las nuevas columnas:")
display(df_with_geolocation[['latitud_normalized', 'longitud_normalized', 'comisaria_mas_cercana', 'distancia_a_comisaria_metros']].head())

Calculando la comisaría más cercana y la distancia para cada delito... Esto puede tardar varios minutos.
Cálculo completado. Mostrando las primeras filas con las nuevas columnas:


,latitud_normalized,longitud_normalized,comisaria_mas_cercana,distancia_a_comisaria_metros
469,-34.726248,-58.283650,Comisaría Vecinal 4-D,12113.888297
470,-34.625121,-58.348646,Comisaría Vecinal 4-C,1487.673080
471,-34.625121,-58.348646,Comisaría Vecinal 4-C,1487.673080
472,-34.625121,-58.348646,Comisaría Vecinal 4-C,1487.673080
473,-34.633645,-58.353495,Comisaría Vecinal 4-C,662.147151


In [ ]:
new_output_filename = 'delitos_2023_geolocated_with_comisarias.csv'
df_with_geolocation.to_csv(new_output_filename, index=False)
print(f"DataFrame con geolocalización y comisaría más cercana guardado en {new_output_filename}")

DataFrame con geolocalización y comisaría más cercana guardado en delitos_2023_geolocated_with_comisarias.csv


El archivo `delitos_2023_geolocated_with_comisarias.csv` ha sido creado y contiene los delitos con sus coordenadas estandarizadas, la comisaría más cercana y la distancia a la misma.

## Resumen de Tareas y Hallazgos

### 1. Carga e Inspección Inicial de `delitos_2023.csv`

Se cargó el conjunto de datos de delitos, que originalmente contenía **155,897** entradas. Se realizó una inspección inicial y se identificaron las columnas `latitud` y `longitud` como las que contienen la información de geolocalización. Se observó que existían valores nulos en estas columnas (348 entradas inicialmente sin latitud/longitud), lo que indicaba la necesidad de separar los delitos con y sin geolocalización, además de los `0.0` como "sin geolocalización".

### 2. Normalización y Estandarización de Geocoordenadas en `delitos_2023.csv`

Esta fue una tarea crucial debido a la naturaleza no estandarizada de las coordenadas. Se detectaron valores atípicos extremos, incluyendo `0.0` y coordenadas con escalas incorrectas (por ejemplo, multiplicadas por $10^5$ o $10^7$). Se desarrolló y aplicó una función `normalize_ba_coords` para:
*   Convertir `0.0` a valores nulos (sin geolocalización).
*   Aplicar divisiones por factores comunes (`1,000,000`, `10,000,000`, etc.) a coordenadas fuera del rango esperado de Buenos Aires para intentar normalizarlas.
*   Asignar `NaN` a aquellas coordenadas que no pudieron ser normalizadas o estaban fuera de un rango plausible.

Tras este proceso, se obtuvieron **149,852** delitos con geolocalización estandarizada y **6,045** entradas fueron clasificadas como "sin geolocalización" (incluyendo los originalmente nulos, los `0.0` y aquellos que no pudieron ser normalizados).

### 3. Inspección y Estandarización de `comisarias_policia.csv`

Se cargó el dataset de comisarías, que consta de **75** estaciones de policía. Se extrajeron las coordenadas de la columna `geometry`. A diferencia de los datos de delitos, todas las **75** coordenadas de las comisarías se encontraron ya en un formato decimal estándar y plausible, dentro del rango geográfico de Buenos Aires, por lo que no requirieron el mismo proceso de normalización de escala.

### 4. Cálculo de Comisaría Más Cercana y Distancia para cada Delito

Para cada delito con geolocalización estandarizada (**149,852** entradas), se calculó la comisaría más cercana y la distancia (en metros) a la misma utilizando la librería `geopy` y el algoritmo de distancia geodésica. Este proceso implicó comparar cada ubicación de delito con las **75** comisarías, resultando en dos nuevas columnas en el DataFrame de delitos geolocalizados: `comisaria_mas_cercana` y `distancia_a_comisaria_metros`.

### 5. Exportación de Datos Limpios y Enriquecidos

Finalmente, se exportó el DataFrame de delitos con geolocalización estandarizada y las nuevas columnas de comisaría más cercana y distancia a un nuevo archivo CSV llamado `delitos_2023_geolocated_with_comisarias.csv`.

In [ ]:
output_filename = 'delitos_2023_geolocated.csv'
df_with_geolocation.to_csv(output_filename, index=False)
print(f"DataFrame with valid geolocation saved to {output_filename}")

DataFrame with valid geolocation saved to delitos_2023_geolocated.csv


The file `delitos_2023_geolocated.csv` has been created and contains only the rows with valid and standardized latitude and longitude values.

In [ ]:
print("Descriptive statistics for 'latitud' in df_with_geolocation:")
display(df_with_geolocation['latitud'].describe())

print("\nDescriptive statistics for 'longitud' in df_with_geolocation:")
display(df_with_geolocation['longitud'].describe())

Descriptive statistics for 'latitud' in df_with_geolocation:


,latitud
count,1.527700e+05
mean,-1.359965e+12
std,2.169116e+14
min,-3.465767e+16
25%,-3.465196e+07
50%,-3.461602e+07
75%,-3.458817e+07
max,-3.459000e+01



Descriptive statistics for 'longitud' in df_with_geolocation:


,longitud
count,1.527700e+05
mean,-9.528928e+08
std,2.099617e+09
min,-5.896677e+10
25%,-5.849417e+07
50%,-5.844037e+07
75%,-5.839324e+07
max,-5.847000e+01


The descriptive statistics confirm that many coordinates are not in a standard format. Typical latitude values for Buenos Aires are around -34 and longitude around -58. We can identify potential outliers by checking values outside these typical ranges and then try to standardize them. It seems some values might be multiplied by $10^5$, $10^6$, or $10^7$ (e.g., -34651960.0 instead of -34.651960).

In [ ]:
# Define typical ranges for Buenos Aires (approximate)
lat_min_ba, lat_max_ba = -35.0, -34.0  # Approx. range for Buenos Aires
lon_min_ba, lon_max_ba = -59.0, -58.0  # Approx. range for Buenos Aires

# Identify coordinates that are clearly out of typical range
outlier_lat = df_with_geolocation[
    (df_with_geolocation['latitud'] < lat_min_ba) |
    (df_with_geolocation['latitud'] > lat_max_ba)
]

outlier_lon = df_with_geolocation[
    (df_with_geolocation['longitud'] < lon_min_ba) |
    (df_with_geolocation['longitud'] > lon_max_ba)
]

print("\n--- Sample of 'latitud' outliers (first 10 unique values) ---")
display(outlier_lat['latitud'].value_counts().head(10))

print("\n--- Sample of 'longitud' outliers (first 10 unique values) ---")
display(outlier_lon['longitud'].value_counts().head(10))

# Additionally, check the overall range of these 'outliers' to see if there's a common scaling factor
print("\nOverall min/max of 'latitud' in df_with_geolocation:")
print(f"Latitud min: {df_with_geolocation['latitud'].min()}, max: {df_with_geolocation['latitud'].max()}")
print("\nOverall min/max of 'longitud' in df_with_geolocation:")
print(f"Longitud min: {df_with_geolocation['longitud'].min()}, max: {df_with_geolocation['longitud'].max()}")


--- Sample of 'latitud' outliers (first 10 unique values) ---


,count
latitud,
-34654819.0,363
-34671152.0,309
-34674789.0,277
-34581731.0,273
-34604562.0,237
-34626368.0,158
-34610119.0,135
-34630705.0,111
-34594442.0,108



--- Sample of 'longitud' outliers (first 10 unique values) ---


,count
longitud,
-58398481.0,364
-58490711.0,309
-58466394.0,277
-58381263.0,273
-58405475.0,233
-58405971.0,132
-58469639.0,113
-58402354.0,107
-58475488.0,100



Overall min/max of 'latitud' in df_with_geolocation:
Latitud min: -3.4657667e+16, max: -34.59

Overall min/max of 'longitud' in df_with_geolocation:
Longitud min: -58966768205.0, max: -58.47
